In [1]:
REPO_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"
DATA_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/YahooAnswer"

print("DATA_DIR:", DATA_DIR)
print("REPO_DIR:", REPO_DIR)


DATA_DIR: /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/YahooAnswer
REPO_DIR: /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM


In [2]:
import os
import numpy as np
import scipy.sparse as sp

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_bow(path):
    X = sp.load_npz(path).astype(np.float32)  
    return X.toarray()                        


def load_word_embeddings(path):
    W = sp.load_npz(path)                 
    return W.astype(np.float32).toarray() 


train_bow = load_bow(os.path.join(DATA_DIR, "train_bow.npz"))
test_bow  = load_bow(os.path.join(DATA_DIR, "test_bow.npz"))
vocab     = load_vocab(os.path.join(DATA_DIR, "vocab.txt"))
word_emb  = load_word_embeddings(os.path.join(DATA_DIR, "word_embeddings.npz"))

print("train_bow:", train_bow.shape)
print("test_bow :", test_bow.shape)
print("word_emb :", word_emb.shape)
print("vocab    :", len(vocab))


train_bow: (10000, 5000)
test_bow : (2500, 5000)
word_emb : (5000, 200)
vocab    : 5000


 Setup + seed

In [3]:
!pip install torch torchvision torchaudio


In [4]:
import os, random
import numpy as np
import torch

REPO_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM" 
DATA_ROOT = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data"
OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECRTM"
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  
    return seed

SEED = set_seed(42)
print("SEED =", SEED)

SEED = 42


Map dataset → YAML (leurs configs)

In [5]:
CONFIGS = {
  "20NG": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_20NG.yaml",
  "AGNews": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_AGNews.yaml",
  "IMBD": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_IMDB.yaml",
  "YahooAnswer": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_YahooAnswer.yaml",
}


Imports modèle + utilitaires

In [6]:
import sys
import yaml
import scipy.io
import scipy.sparse as sp
from types import SimpleNamespace
from torch.utils.data import DataLoader, TensorDataset

sys.path.append(REPO_DIR)
from models.ECRTM import ECRTM

def load_cfg(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)  

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_bow_npz(path):
    X = sp.load_npz(path).astype(np.float32)  
    return X.toarray()                       

def load_word_emb_npz(path):
    W = sp.load_npz(path)                 
    return W.astype(np.float32).toarray()

Fonctions: build args + infer theta

In [7]:
def build_args_from_yaml(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)  # copie
    cfg["num_topic"] = K
    cfg["vocab_size"] = vocab_size
    cfg["word_embeddings"] = word_emb
    return SimpleNamespace(**cfg)

def infer_theta(model, bow_np, device, batch_size=256):
    model.eval()
    thetas = []
    with torch.no_grad():
        X = torch.from_numpy(bow_np).float()
        loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=False)
        for (bow,) in loader:
            bow = bow.to(device)
            theta = model.get_theta(bow)  
            thetas.append(theta.cpu().numpy())
    return np.vstack(thetas)


 run complet: train_one(dataset, K)

In [8]:
from tqdm.auto import tqdm

def train_one(dataset, K, seed=42):
    set_seed(seed)

    data_dir = os.path.join(DATA_ROOT, dataset)
    cfg = load_cfg(CONFIGS[dataset])

    # Hyperparams depuis YAML
    epochs = int(cfg["epochs"])
    batch_size = int(cfg["batch_size"])
    lr = float(cfg["learning_rate"])

    # Data
    train_bow = load_bow_npz(os.path.join(data_dir, "train_bow.npz"))
    test_bow  = load_bow_npz(os.path.join(data_dir, "test_bow.npz"))
    vocab     = load_vocab(os.path.join(data_dir, "vocab.txt"))
    word_emb  = load_word_emb_npz(os.path.join(data_dir, "word_embeddings.npz"))

    V = train_bow.shape[1]
    assert V == test_bow.shape[1] == len(vocab) == word_emb.shape[0], "Mismatch vocab sizes"


    args = build_args_from_yaml(cfg, K=K, vocab_size=V, word_emb=word_emb)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ECRTM(args).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    X = torch.from_numpy(train_bow).float()
    loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"{dataset} | K={K} | seed={seed}", position=0)
    for epoch in epoch_bar:
        model.train()
        total_loss = 0.0

        batch_bar = tqdm(loader, desc=f"epoch {epoch}/{epochs}", leave=False, position=1)
        for (bow,) in batch_bar:
            bow = bow.to(device)
            rst = model(bow)
            loss = rst["loss"]

            opt.zero_grad()
            loss.backward()
            opt.step()

            l = float(loss.detach().cpu())
            total_loss += l
            batch_bar.set_postfix(loss=l)

        epoch_bar.set_postfix(avg_loss=total_loss / len(loader))

    model.eval()
    with torch.no_grad():
        beta = model.get_beta().cpu().numpy()

    train_theta = infer_theta(model, train_bow, device=device, batch_size=256)
    test_theta  = infer_theta(model, test_bow,  device=device, batch_size=256)

    out_path = os.path.join(OUT_DIR, f"{dataset}_ECRTM_K{K}_seed{seed}.mat")
    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    print("Saved:", out_path)
    return out_path


20NG

In [ ]:
#for K in [20, 50, 100]:
#    train_one("20NG", K=K, seed=42)


IMDB

In [ ]:
#for K in [20, 50, 100]:
    #train_one("IMDB", K=K, seed=42)


AGNews

In [ ]:
for K in [20, 50, 100]:
    train_one("AGNews", K=K, seed=42)


YahooAnswer

In [9]:
for K in [20, 50, 100]:
    train_one("YahooAnswer", K=K, seed=42)


YahooAnswer | K=20 | seed=42:   0%|          | 0/500 [00:00<?, ?it/s]

epoch 1/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 2/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 3/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 4/500:   0%|          | 0/50 [00:00<?, ?it/s]

...

epoch 497/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 498/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 499/500:   0%|          | 0/50 [00:00<?, ?it/s]

epoch 500/500:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECTRM/YahooAnswer_ECRTM_K100_seed42.mat


 Imports + dossier de sortie LDA

In [10]:
import os
import numpy as np
import scipy.io
import scipy.sparse as sp
from sklearn.decomposition import LatentDirichletAllocation

RUNS_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"  
os.makedirs(RUNS_DIR, exist_ok=True)
print("RUNS_DIR =", RUNS_DIR)

OUT_DIR_LDA = os.path.join(RUNS_DIR, "LDA")
os.makedirs(OUT_DIR_LDA, exist_ok=True)
print("OUT_DIR_LDA =", OUT_DIR_LDA)

/Users/hydersidra/opt/anaconda3/lib/python3.9/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


RUNS_DIR = /Users/hydersidra/Documents/Master/PPD/ECRTM_runs
OUT_DIR_LDA = /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA


Loader sparse (spécifique LDA)

In [11]:
def load_bow_sparse(path):
    return sp.load_npz(path)


Fonction train_one_lda(dataset, K)

In [13]:
def train_one_lda(dataset, K, seed=42, max_iter=50):
    data_dir = os.path.join(DATA_ROOT, dataset)

    X_train = load_bow_sparse(os.path.join(data_dir, "train_bow.npz"))
    X_test  = load_bow_sparse(os.path.join(data_dir, "test_bow.npz"))

    out_path = os.path.join(OUT_DIR_LDA, f"{dataset}_LDA_K{K}_seed{seed}.mat")
    if os.path.exists(out_path):
        print("Skip (already exists):", out_path)
        return out_path

    lda = LatentDirichletAllocation(
        n_components=K,           
        max_iter=max_iter,        
        learning_method="batch",
        random_state=seed,
        verbose=1                 
    )
    lda.fit(X_train) 

    beta = lda.components_.astype(np.float32)          
    beta = beta / beta.sum(axis=1, keepdims=True)

    train_theta = lda.transform(X_train).astype(np.float32)  
    test_theta  = lda.transform(X_test).astype(np.float32)   

    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    print("Saved:", out_path)
    return out_path


20NG

In [25]:
for K in [20, 50, 100]:
    train_one_lda("20NG", K=K, seed=42, max_iter=50)


Skip (already exists): /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/20NG_LDA_K20_seed42.mat
Skip (already exists): /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/20NG_LDA_K50_seed42.mat
Skip (already exists): /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/20NG_LDA_K100_seed42.mat


AGNews

In [ ]:
for K in [20, 50, 100]:
    train_one_lda("AGNews", K=K, seed=42, max_iter=50)


YahooAnswer

In [14]:
for K in [20, 50, 100]:
    train_one_lda("YahooAnswer", K=K, seed=42, max_iter=50)


iteration: 1 of max_iter: 50
iteration: 2 of max_iter: 50
iteration: 3 of max_iter: 50
iteration: 4 of max_iter: 50
iteration: 5 of max_iter: 50
iteration: 6 of max_iter: 50
iteration: 7 of max_iter: 50
iteration: 8 of max_iter: 50
iteration: 9 of max_iter: 50
iteration: 10 of max_iter: 50
iteration: 11 of max_iter: 50
iteration: 12 of max_iter: 50
iteration: 13 of max_iter: 50
iteration: 14 of max_iter: 50
iteration: 15 of max_iter: 50
iteration: 16 of max_iter: 50
iteration: 17 of max_iter: 50
iteration: 18 of max_iter: 50
iteration: 19 of max_iter: 50
iteration: 20 of max_iter: 50
iteration: 21 of max_iter: 50
iteration: 22 of max_iter: 50
iteration: 23 of max_iter: 50
iteration: 24 of max_iter: 50
iteration: 25 of max_iter: 50
iteration: 26 of max_iter: 50
iteration: 27 of max_iter: 50
iteration: 28 of max_iter: 50
iteration: 29 of max_iter: 50
iteration: 30 of max_iter: 50
iteration: 31 of max_iter: 50
iteration: 32 of max_iter: 50
iteration: 33 of max_iter: 50
iteration: 34 of ma

In [33]:
for K in [20, 50, 100]:
    train_one_lda("YahooAnswer", K=K, seed=42, max_iter=200)


iteration: 1 of max_iter: 200
iteration: 2 of max_iter: 200
iteration: 3 of max_iter: 200
iteration: 4 of max_iter: 200
iteration: 5 of max_iter: 200
iteration: 6 of max_iter: 200
iteration: 7 of max_iter: 200
iteration: 8 of max_iter: 200
iteration: 9 of max_iter: 200
iteration: 10 of max_iter: 200
iteration: 11 of max_iter: 200
iteration: 12 of max_iter: 200
iteration: 13 of max_iter: 200
iteration: 14 of max_iter: 200
iteration: 15 of max_iter: 200
iteration: 16 of max_iter: 200
iteration: 17 of max_iter: 200
iteration: 18 of max_iter: 200
iteration: 19 of max_iter: 200
iteration: 20 of max_iter: 200
iteration: 21 of max_iter: 200
iteration: 22 of max_iter: 200
iteration: 23 of max_iter: 200
iteration: 24 of max_iter: 200
iteration: 25 of max_iter: 200
iteration: 26 of max_iter: 200
iteration: 27 of max_iter: 200
iteration: 28 of max_iter: 200
iteration: 29 of max_iter: 200
iteration: 30 of max_iter: 200
iteration: 31 of max_iter: 200
iteration: 32 of max_iter: 200
iteration: 33 of 

KeyboardInterrupt: 

In [39]:
import os
import numpy as np
import scipy.io 

# CHEMINS
PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"
OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"
OUT_DIR_LDA = f"{OUT_DIR}/LDA"

DATASET = "YahooAnswer"  
VOCAB_PATH = os.path.join(PROJECT_DIR, f"data/{DATASET}/vocab.txt")

METRICS_DIR = os.path.join(PROJECT_DIR, "metrics")
os.makedirs(METRICS_DIR, exist_ok=True)

# Chargé VOCABULAIRE
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

print(f" Vocab chargé : {len(vocab)} mots")

# Configuration
TOPN = 15  
configs = [
    ("LDA", 20, 42),
    ("LDA", 50, 42),
    ("LDA", 100, 42),
]

# generer les topics files 
for model, K, seed in configs:
    # Charger le .mat
    mat_file = f"{DATASET}_{model}_K{K}_seed{seed}.mat"
    mat_path = os.path.join(OUT_DIR_LDA, mat_file)
    
    if not os.path.isfile(mat_path):
        print(f"MANQUANT: {mat_path}")
        continue
    
    loaded_data = scipy.io.loadmat(mat_path)
    beta = loaded_data["beta"]  # (K, V)
    
    # Générer le fichier topics
    topics_file = os.path.join(METRICS_DIR, f"topics_{DATASET}_{model}_K{K}_seed{seed}.txt")
    
    with open(topics_file, "w", encoding="utf-8") as f:
        for k in range(beta.shape[0]):
            top_idx = np.argsort(beta[k])[::-1][:TOPN]
            f.write(" ".join(vocab[i] for i in top_idx) + "\n")  # ← FIX: \n pas \\n
    
    print(f"Topics sauvegardés : {topics_file}")

print("\n Terminé !")


 Vocab chargé : 5000 mots
Topics sauvegardés : /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/metrics/topics_YahooAnswer_LDA_K20_seed42.txt
Topics sauvegardés : /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/metrics/topics_YahooAnswer_LDA_K50_seed42.txt
Topics sauvegardés : /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/metrics/topics_YahooAnswer_LDA_K100_seed42.txt

 Terminé !


PALMETTO

In [16]:
WIKI_DIR="/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd"
!mkdir -p "$WIKI_DIR"

!ls -l "$WIKI_DIR/wikipedia_bd"

total 13422928
-rw-r--r--@ 1 hydersidra  staff         310 21 mai  2014 _3f5.cfe
-rw-r--r--@ 1 hydersidra  staff    10708619 21 mai  2014 _3f5.cfs
-rw-r--r--@ 1 hydersidra  staff         285 21 mai  2014 _3f5.si
-rw-r--r--@ 1 hydersidra  staff   928213930 21 mai  2014 _3gj_Lucene41_0.doc
-rw-r--r--@ 1 hydersidra  staff  1464777604 21 mai  2014 _3gj_Lucene41_0.pos
-rw-r--r--@ 1 hydersidra  staff   154487950 21 mai  2014 _3gj_Lucene41_0.tim
-rw-r--r--@ 1 hydersidra  staff     3327506 21 mai  2014 _3gj_Lucene41_0.tip
-rw-r--r--@ 1 hydersidra  staff    11999360 21 mai  2014 _3gj.fdt
-rw-r--r--@ 1 hydersidra  staff       41205 21 mai  2014 _3gj.fdx
-rw-r--r--@ 1 hydersidra  staff         125 21 mai  2014 _3gj.fnm
-rw-r--r--@ 1 hydersidra  staff     3069026 21 mai  2014 _3gj.nvd
-rw-r--r--@ 1 hydersidra  staff          46 21 mai  2014 _3gj.nvm
-rw-r--r--@ 1 hydersidra  staff         409 21 mai  2014 _3gj.si
-rw-r--r--@ 1 hydersidra  staff  2746741666 21 mai  2014 _3gj.tvd
-rw-r--r--@ 1 hyder

In [17]:
!test -f "$PALMETTO_JAR" && echo "OK: jar" || echo "KO: jar"
!test -d "$WIKI_BD_DIR" && echo "OK: wikipedia_bd dir" || echo "KO: wikipedia_bd dir"
!test -f "$TOPICS_FILE" && echo "OK: topics file" || echo "KO: topics file"

!ls -lah "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd" | head -n 50


KO: jar
KO: wikipedia_bd dir
OK: topics file
total 352
drwx------@   5 hydersidra  staff   160B 26 jan 11:15 .
drwxr-xr-x@   5 hydersidra  staff   160B 26 jan 11:15 ..
-rw-r--r--@   1 hydersidra  staff   8,0K 26 jan 11:16 .DS_Store
drwxr-xr-x@ 112 hydersidra  staff   3,5K 26 jan 11:15 wikipedia_bd
-rw-r--r--@   1 hydersidra  staff   162K 24 avr  2014 wikipedia_bd.histogram


In [35]:
import os
import numpy as np
import scipy.io
import subprocess

#  Chemins 
PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"

DATASET = "YahooAnswer"
VOCAB_PATH = os.path.join(PROJECT_DIR, f"data/{DATASET}/vocab.txt")

OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"
OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"

os.makedirs(OUT_DIR_LDA, exist_ok=True)
os.makedirs(OUT_DIR_ECRTM, exist_ok=True)

# Palmetto 
PALMETTO_JAR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/palmetto-0.1.0-jar-with-dependencies.jar"
WIKI_BD_DIR  = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd/wikipedia_bd"
TOPN = 15

# Vocab
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

print(f"Vocab chargé: {len(vocab)} mots\n")

# Fichiers .mat 
lda_files = [
    "YahooAnswer_LDA_K20_seed42.mat",
    "YahooAnswer_LDA_K50_seed42.mat",
    "YahooAnswer_LDA_K100_seed42.mat",
]

ecrtm_files = [
    "YahooAnswer_ECRTM_K20_seed42.mat",
    "YahooAnswer_ECRTM_K50_seed42.mat",
    "YahooAnswer_ECRTM_K100_seed42.mat",
]

#  Fonction de processing 
def process_mat_files(mat_files, base_dir, model_name):
    for mat_name in mat_files:
        mat_path = os.path.join(base_dir, mat_name)
        
        if not os.path.isfile(mat_path):
            print(f" Introuvable: {mat_path}")
            continue
        
        loaded = scipy.io.loadmat(mat_path)
        if "beta" not in loaded:
            print(f"'beta' introuvable dans {mat_name}. Keys: {list(loaded.keys())}")
            continue
        
        beta = loaded["beta"]  # (K, V)
        
        base_name = mat_name.replace(".mat", "")
        
        # Sauvegarde 
        topics_file = os.path.join(base_dir, f"topics_for_cv_{base_name}.txt")
        cv_out_file = os.path.join(base_dir, f"palmetto_CV_{base_name}.txt")
        
        # Écrire topics (top TOPN mots)
        with open(topics_file, "w", encoding="utf-8") as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                # ← FIX ICI : \n au lieu de \\n
                f.write(" ".join(vocab[i] for i in top_idx) + "\n")
        
        # Lancer Palmetto
        cmd = ["java", "-jar", PALMETTO_JAR, WIKI_BD_DIR, "C_V", topics_file]
        try:
            out = subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)
            with open(cv_out_file, "w", encoding="utf-8") as f:
                f.write(out)
            print(f"{model_name} OK: {mat_name} -> {cv_out_file}")
        except subprocess.CalledProcessError as e:
            print(f" Erreur Palmetto pour {mat_name}:\n{e.output}")


print("=== Processing LDA ===")
process_mat_files(lda_files, OUT_DIR_LDA, "LDA")

print("\n=== Processing ECRTM ===")
process_mat_files(ecrtm_files, OUT_DIR_ECRTM, "ECRTM")

print("\n Terminé. Résultats dans:")
print(f"  LDA   → {OUT_DIR_LDA}")
print(f"  ECRTM → {OUT_DIR_ECRTM}")


✅ Vocab chargé: 5000 mots

=== Processing LDA ===
✅ LDA OK: YahooAnswer_LDA_K20_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/palmetto_CV_YahooAnswer_LDA_K20_seed42.txt
✅ LDA OK: YahooAnswer_LDA_K50_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/palmetto_CV_YahooAnswer_LDA_K50_seed42.txt
✅ LDA OK: YahooAnswer_LDA_K100_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/LDA/palmetto_CV_YahooAnswer_LDA_K100_seed42.txt

=== Processing ECRTM ===
✅ ECRTM OK: YahooAnswer_ECRTM_K20_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECRTM/palmetto_CV_YahooAnswer_ECRTM_K20_seed42.txt
✅ ECRTM OK: YahooAnswer_ECRTM_K50_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECRTM/palmetto_CV_YahooAnswer_ECRTM_K50_seed42.txt
✅ ECRTM OK: YahooAnswer_ECRTM_K100_seed42.mat -> /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECRTM/palmetto_CV_YahooAnswer_ECRTM_K100_seed42.txt

✅ Terminé. Résultats dans:
  LDA   → /Users/hyder

In [36]:
import os
import numpy as np
import pandas as pd

OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"
OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"

DATASET = "YahooAnswer"  

configs = [
    ("LDA",   20, 42, OUT_DIR_LDA),
    ("LDA",   50, 42, OUT_DIR_LDA),
    ("LDA",  100, 42, OUT_DIR_LDA),
    ("ECRTM", 20, 42, OUT_DIR_ECRTM),
    ("ECRTM", 50, 42, OUT_DIR_ECRTM),
    ("ECRTM",100, 42, OUT_DIR_ECRTM),
]

rows = []

for model, K, seed, folder in configs:
    fn = f"palmetto_CV_{DATASET}_{model}_K{K}_seed{seed}.txt"
    path = os.path.join(folder, fn)
    
    if not os.path.isfile(path):
        print(f" MANQUANT: {path}")
        continue
    
    print(f" Lecture: {fn}")
    
    vals = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line or "INFO" in line or "ERROR" in line or line.startswith("2026-"):
                continue
            
          
            parts = line.split('\t')
            
            if len(parts) >= 2:
                score_str = parts[1].replace(',', '.')
                try:
                    score = float(score_str)
                    vals.append(score)
                except ValueError:
                    pass
    
    if not vals:
        print(f" Aucun score trouvé dans {fn}")
        continue
    
    vals = np.array(vals, dtype=float)
    
    rows.append({
        "model": model,
        "K": K,
        "seed": seed,
        "CV_mean": float(vals.mean()),
        "CV_std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
        "CV_min": float(vals.min()),
        "CV_max": float(vals.max()),
        "n_topics": len(vals),
    })

if not rows:
    print("\n Aucun score C_V trouvé.")
else:
    df_cv = pd.DataFrame(rows).sort_values(["model", "K"])
    
   
    out_csv = os.path.join(OUT_DIR, f"CV_summary_{DATASET}.csv")
    df_cv.to_csv(out_csv, index=False)
    
    print(f"\n Résumé sauvegardé: {out_csv}\n")
    print(df_cv.to_string(index=False))


 Lecture: palmetto_CV_YahooAnswer_LDA_K20_seed42.txt
 Lecture: palmetto_CV_YahooAnswer_LDA_K50_seed42.txt
 Lecture: palmetto_CV_YahooAnswer_LDA_K100_seed42.txt
 Lecture: palmetto_CV_YahooAnswer_ECRTM_K20_seed42.txt
 Lecture: palmetto_CV_YahooAnswer_ECRTM_K50_seed42.txt
 Lecture: palmetto_CV_YahooAnswer_ECRTM_K100_seed42.txt

 Résumé sauvegardé: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/CV_summary_YahooAnswer.csv

model   K  seed  CV_mean   CV_std  CV_min  CV_max  n_topics
ECRTM  20    42 0.371946 0.059974 0.21740 0.46958        20
ECRTM  50    42 0.402173 0.088469 0.23892 0.67203        50
ECRTM 100    42 0.386054 0.081806 0.21311 0.59807       100
  LDA  20    42 0.342713 0.073870 0.25550 0.49704        20
  LDA  50    42 0.350948 0.066223 0.24601 0.52937        50
  LDA 100    42 0.355209 0.060193 0.23979 0.50513       100


In [41]:
import pandas as pd

data = {
    'Modèle': ['ECRTM', 'ECRTM', 'LDA', 'LDA'],
    'K': [50, 100, 50, 100],
    'Notre CV Mean': [0.402, 0.386, 0.350, 0.355],
    'Papier CV Mean': [0.405, 0.389, 0.359, 0.359],
}

df = pd.DataFrame(data)

df['Écart'] = (df['Notre CV Mean'] - df['Papier CV Mean']).round(3).apply(lambda x: f"{x:+.3f}")
df['Notre CV Mean'] = df['Notre CV Mean'].round(3)
df['Papier CV Mean'] = df['Papier CV Mean'].round(3)

df_display = df[['Modèle', 'K', 'Notre CV Mean', 'Papier CV Mean', 'Écart']]
df_display


,Modèle,K,Notre CV Mean,Papier CV Mean,Écart
0,ECRTM,50,0.402,0.405,-0.003
1,ECRTM,100,0.386,0.389,-0.003
2,LDA,50,0.350,0.359,-0.009
3,LDA,100,0.355,0.359,-0.004


In [24]:
ls -lh /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/


total 0
drwxr-xr-x@ 10 hydersidra  staff   320B 26 jan 11:15 20NG/
drwxr-xr-x@ 10 hydersidra  staff   320B 26 jan 11:15 AGNews/
drwxr-xr-x@ 10 hydersidra  staff   320B 26 jan 11:15 IMDB/
drwxr-xr-x@  5 hydersidra  staff   160B 26 jan 11:15 stopwords/
drwxr-xr-x@ 10 hydersidra  staff   320B 26 jan 11:15 YahooAnswer/
drwxr-xr-x@ 10 hydersidra  staff   320B 26 jan 11:15 Yelp/


In [40]:
import os
import numpy as np
import pandas as pd
import scipy.io
from sklearn.metrics import normalized_mutual_info_score

#  CHEMINS 
PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project"
OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"

OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"

VOCAB_PATH  = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/YahooAnswer/vocab.txt"
LABELS_PATH = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/YahooAnswer/test_labels.txt"

TOPN_TD = 15  # Topic Diversity sur top-25 mots (standard papiers)

#  DATA
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

y_true = np.loadtxt(LABELS_PATH, dtype=int)
print(f"Chargé: {len(vocab)} mots vocab, {len(y_true)} documents avec labels\n")

#  METRICS 
def topic_diversity_from_beta(beta, vocab, topn=25):
    """TD = proportion de mots uniques dans les top-N de tous les topics"""
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(vocab[i] for i in top_idx)
    return len(set(all_words)) / max(1, len(all_words))

def purity_score(y_true, y_pred):
    """Purity = proportion de docs dans le cluster majoritaire pour chaque cluster"""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    purity = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) == 0:
            continue
        _, counts = np.unique(y_true[idx], return_counts=True)
        purity += counts.max()
    return purity / len(y_true)

def find_theta_in_mat(d, K, N):
    """Cherche la matrice doc-topic (N, K) dans le .mat"""
    skip = {"__header__", "__version__", "__globals__"}
    for name, arr in d.items():
        if name in skip or not isinstance(arr, np.ndarray) or arr.ndim != 2:
            continue
        if arr.shape == (N, K):
            return name, arr
        if arr.shape == (K, N):
            return name, arr.T
    raise ValueError(f"Theta (doc-topic) introuvable. Clés disponibles: {list(d.keys())}")

#  CONFIG 
configs = [
    ("LDA",   20, 42, OUT_DIR_LDA),
    ("LDA",   50, 42, OUT_DIR_LDA),
    ("LDA",  100, 42, OUT_DIR_LDA),
    ("ECRTM", 20, 42, OUT_DIR_ECRTM),
    ("ECRTM", 50, 42, OUT_DIR_ECRTM),
    ("ECRTM",100, 42, OUT_DIR_ECRTM),
]

rows = []

for model, K, seed, folder in configs:
    fn = f"YahooAnswer_{model}_K{K}_seed{seed}.mat"
    mat_path = os.path.join(folder, fn)
    
    if not os.path.isfile(mat_path):
        print(f" MANQUANT: {mat_path}")
        continue
    
    print(f" Lecture: {fn}")
    
    d = scipy.io.loadmat(mat_path)
    beta = d["beta"]  # (K, V)
    
    # Trouver theta (doc-topic)
    try:
        theta_key, theta = find_theta_in_mat(d, K, len(y_true))
    except ValueError as e:
        print(f" Erreur pour {fn}: {e}")
        continue
    
    # Clustering: chaque doc assigné au topic dominant
    y_pred = theta.argmax(axis=1)
    
    # Métriques
    td = topic_diversity_from_beta(beta, vocab, topn=TOPN_TD)
    purity = purity_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    
    rows.append({
        "model": model,
        "K": K,
        "seed": seed,
        "TD": round(td, 4),
        "Purity": round(purity, 4),
        "NMI": round(nmi, 4),
    })

#  SAVE & DISPLAY
if not rows:
    print("\nAucune métrique calculée.")
else:
    df = pd.DataFrame(rows).sort_values(["model", "K"])
    
    # ← CHANGÉ : nom du CSV avec DATASET
    out_csv = os.path.join(PROJECT_DIR, "metrics", f"TD_Purity_NMI_{DATASET}.csv")
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)
    
    print(f"\nRésumé sauvegardé: {out_csv}\n")
    print(df.to_string(index=False))


Chargé: 5000 mots vocab, 2500 documents avec labels

 Lecture: YahooAnswer_LDA_K20_seed42.mat
 Lecture: YahooAnswer_LDA_K50_seed42.mat
 Lecture: YahooAnswer_LDA_K100_seed42.mat
 Lecture: YahooAnswer_ECRTM_K20_seed42.mat
 Lecture: YahooAnswer_ECRTM_K50_seed42.mat
 Lecture: YahooAnswer_ECRTM_K100_seed42.mat

Résumé sauvegardé: /Users/hydersidra/Documents/Master/PPD/ECRTM_project/metrics/TD_Purity_NMI_YahooAnswer.csv

model   K  seed     TD  Purity    NMI
ECRTM  20    42 0.9967  0.5452 0.3185
ECRTM  50    42 1.0000  0.5288 0.2912
ECRTM 100    42 0.9080  0.5580 0.3077
  LDA  20    42 0.7133  0.4472 0.2542
  LDA  50    42 0.7440  0.4632 0.2485
  LDA 100    42 0.7400  0.4484 0.2317


In [42]:
import pandas as pd

data = {
    'Modèle': ['LDA', 'LDA', 'ECRTM', 'ECRTM'],
    'K': [50, 100, 50, 100],
    'TD Papier': [0.843, 0.602, 0.985, 0.903],
    'Pureté Papier': [0.288, 0.297, 0.550, 0.563],
    'NMI Papier': [0.144, 0.148, 0.295, 0.311],
    'Notre TD': [0.7440, 0.7400, 1.0000, 0.9080],  
    'Notre Pureté': [0.4632, 0.4484, 0.5288, 0.5580], 
    'Notre NMI': [0.2485, 0.2317, 0.2912, 0.3077]
}

df = pd.DataFrame(data)

df['Écart TD'] = (df['Notre TD'] - df['TD Papier']).round(3).apply(lambda x: f"{x:+.3f}")
df['Écart Pureté'] = (df['Notre Pureté'] - df['Pureté Papier']).round(3).apply(lambda x: f"{x:+.3f}")
df['Écart NMI'] = (df['Notre NMI'] - df['NMI Papier']).round(3).apply(lambda x: f"{x:+.3f}")

df_display = df[['Modèle', 'K', 'TD Papier', 'Pureté Papier', 'NMI Papier', 
                'Notre TD', 'Notre Pureté', 'Notre NMI', 'Écart TD', 'Écart Pureté', 'Écart NMI']]

df_display


,Modèle,K,TD Papier,Pureté Papier,NMI Papier,Notre TD,Notre Pureté,Notre NMI,Écart TD,Écart Pureté,Écart NMI
0,LDA,50,0.843,0.288,0.144,0.744,0.4632,0.2485,-0.099,+0.175,+0.105
1,LDA,100,0.602,0.297,0.148,0.740,0.4484,0.2317,+0.138,+0.151,+0.084
2,ECRTM,50,0.985,0.550,0.295,1.000,0.5288,0.2912,+0.015,-0.021,-0.004
3,ECRTM,100,0.903,0.563,0.311,0.908,0.5580,0.3077,+0.005,-0.005,-0.003
